In [1]:
import warnings 
warnings.filterwarnings("ignore")

In [2]:
import os
import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

import torchaudio
import torchaudio.transforms as AT

from transformers import ASTForAudioClassification, ASTFeatureExtractor

# from sklearn.model_selection import StratifiedGroupKFold
# from sklearn.metrics import f1_score, classification_report, confusion_matrix
# import matplotlib.pyplot as plt
# import seaborn as sns

# # import wandb
# # from kaggle_secrets import UserSecretsClient

2026-03-06 07:42:49.087985: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772782969.275662      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772782969.332907      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772782969.755742      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772782969.755788      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772782969.755791      24 computation_placer.cc:177] computation placer alr

In [3]:
# try:
#     secrets = UserSecretsClient()
#     wandb_key = secrets.get_secret("WANDB_API_KEY")
#     wandb.login(key=wandb_key)
#     print("W&B login successful!")
# except Exception as e:
#     print(f"W&B login via secrets failed ({e}), trying anonymous...")
#     wandb.login(anonymous="allow")

In [4]:
AUDIO_DIR = "/kaggle/input/datasets/sreekaranreddy2005/dlgenai-proj-audio-classification/messy-mashup-augmented/audio"
CSV_PATH = "/kaggle/input/datasets/sreekaranreddy2005/dlgenai-proj-audio-classification/messy-mashup-augmented/metadata.csv"
OUTPUT_DIR = "/kaggle/working"
WANDB_PROJECT = "24f2000010-t12026"

In [5]:
#Audio params
TARGET_SR = 16000  
AUDIO_DURATION = 5  

# Training params
BATCH_SIZE = 16       
GRAD_ACCUM_STEPS = 4  
NUM_WORKERS = 2
EPOCHS = 9
LR = 5e-5             
WEIGHT_DECAY = 1e-2
WARMUP_EPOCHS = 2
NUM_CLASSES = 10
SEED = 42
N_FOLDS = 5
TRAIN_FOLD = 0

AST_MODEL_NAME = "MIT/ast-finetuned-audioset-10-10-0.4593"

In [6]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [7]:
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop',
          'jazz', 'metal', 'pop', 'reggae', 'rock']
GENRE_TO_IDX = {g: i for i, g in enumerate(GENRES)}
IDX_TO_GENRE = {i: g for g, i in GENRE_TO_IDX.items()}

In [8]:
# print("Loading metadata...")
# df = pd.read_csv(CSV_PATH)

# if 'filepath' in df.columns:
#     df['filename'] = df['filepath'].apply(lambda x: os.path.basename(x))
# elif 'image' in df.columns:
#     df['filename'] = df['image'].str.replace('.png', '.wav')

# if 'genre' in df.columns:
#     df['labels'] = df['genre']
# elif 'labels' not in df.columns:
#     df.columns = ['filepath', 'labels']
#     df['filename'] = df['filepath'].apply(lambda x: os.path.basename(x))

# df['song_group'] = df['filename'].apply(
#     lambda x: '_'.join(os.path.splitext(x)[0].split('_')[:2])
# )
# df['label_idx'] = df['labels'].map(GENRE_TO_IDX)

# existing_files = set(os.listdir(AUDIO_DIR))
# df = df[df['filename'].isin(existing_files)].reset_index(drop=True)

# print(f"Total samples: {len(df)}")
# print(f"Unique songs:  {df['song_group'].nunique()}")
# print(f"Genre distribution:\n{df['labels'].value_counts()}\n")

In [9]:
# sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
# for fold_idx, (train_idx, val_idx) in enumerate(
#     sgkf.split(df, df['label_idx'], groups=df['song_group'])
# ):
#     if fold_idx == TRAIN_FOLD:
#         train_df = df.iloc[train_idx].reset_index(drop=True)
#         val_df = df.iloc[val_idx].reset_index(drop=True)
#         break

# print(f"Fold {TRAIN_FOLD}: Train={len(train_df)}, Val={len(val_df)}")

In [10]:
print(f"\nLoading AST feature extractor from: {AST_MODEL_NAME}")
feature_extractor = ASTFeatureExtractor.from_pretrained(AST_MODEL_NAME)
print(f"  Sampling rate: {feature_extractor.sampling_rate}")
print(f"  Max length: {feature_extractor.max_length}")


Loading AST feature extractor from: MIT/ast-finetuned-audioset-10-10-0.4593


preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

  Sampling rate: 16000
  Max length: 1024


In [11]:
# wandb.init(
#     project=WANDB_PROJECT,
#     name="model6-ast-pretrained-Resumed",
#     config={
#         "model": "AST",
#         "pretrained": AST_MODEL_NAME,
#         "target_sr": TARGET_SR,
#         "audio_duration": AUDIO_DURATION,
#         "batch_size": BATCH_SIZE,
#         "effective_batch_size": BATCH_SIZE * GRAD_ACCUM_STEPS,
#         "grad_accum_steps": GRAD_ACCUM_STEPS,
#         "epochs": EPOCHS,
#         "lr": LR,
#         "weight_decay": WEIGHT_DECAY,
#         "warmup_epochs": WARMUP_EPOCHS,
#         "num_classes": NUM_CLASSES,
#         "fold": TRAIN_FOLD,
#         "optimizer": "AdamW (layer-wise LR)",
#         "scheduler": "Warmup+Cosine",
#         "augmentations": "VolumPerturb+NoiseInject+TimeShift",
#         "label_smoothing": 0.1,
#     }
# )

In [12]:
# class ASTDataset(Dataset):
#     def __init__(self, dataframe, audio_dir, feature_extractor,
#                  target_sr=16000, duration=3, augment=False):
#         self.df = dataframe
#         self.audio_dir = audio_dir
#         self.feature_extractor = feature_extractor
#         self.target_sr = target_sr
#         self.target_length = target_sr * duration
#         self.augment = augment

#     def __len__(self):
#         return len(self.df)

#     def _load_audio(self, path):
#         waveform, sr = torchaudio.load(path)

#         if waveform.shape[0] > 1:
#             waveform = waveform.mean(dim=0, keepdim=True)

    
#         if sr != self.target_sr:
#             resampler = AT.Resample(sr, self.target_sr)
#             waveform = resampler(waveform)

    
#         if waveform.shape[1] < self.target_length:
#             padding = self.target_length - waveform.shape[1]
#             waveform = torch.nn.functional.pad(waveform, (0, padding))
#         else:
#             waveform = waveform[:, :self.target_length]

#         return waveform.squeeze(0).numpy()  

#     def _augment(self, waveform):
#         """Simple numpy-based augmentation."""
        
#         if random.random() < 0.5:
#             gain = random.uniform(0.7, 1.3)
#             waveform = waveform * gain

    
#         if random.random() < 0.3:
#             noise = np.random.randn(len(waveform)) * 0.005
#             waveform = waveform + noise

    
#         if random.random() < 0.3:
#             shift = random.randint(-1600, 1600)
#             waveform = np.roll(waveform, shift)

#         return waveform.astype(np.float32)

#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]
#         audio_path = os.path.join(self.audio_dir, row['filename'])
#         label = row['label_idx']

#         waveform = self._load_audio(audio_path)

#         if self.augment:
#             waveform = self._augment(waveform)

    
#         inputs = self.feature_extractor(
#             waveform,
#             sampling_rate=self.target_sr,
#             return_tensors="pt",
#             padding="max_length",
#             max_length=self.feature_extractor.max_length
#         )

#         input_values = inputs['input_values'].squeeze(0)  

#         return input_values, label

In [13]:
# sample_input, sample_label = train_dataset[0]
# print(f"AST input shape: {sample_input.shape}")
# print(f"Label: {sample_label} ({IDX_TO_GENRE[sample_label]})")

print(f"\nLoading AST model from: {AST_MODEL_NAME}")
model = ASTForAudioClassification.from_pretrained(
    AST_MODEL_NAME,
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True
)

model.config.id2label = IDX_TO_GENRE
model.config.label2id = GENRE_TO_IDX

model = model.to(DEVICE)

# total_params = sum(p.numel() for p in model.parameters())
# trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
# print(f"Total params:     {total_params:,}")
# print(f"Trainable params: {trainable_params:,}")

# wandb.watch(model, log="gradients", log_freq=50)


Loading AST model from: MIT/ast-finetuned-audioset-10-10-0.4593


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
# def get_optimizer_params(model, base_lr=5e-5, head_lr=1e-3):
#     """Different LR for backbone vs classifier head."""
#     no_decay = ['bias', 'LayerNorm.weight', 'layernorm.weight']

#     optimizer_params = [
#         # Classifier head — higher learning rate
#         {
#             'params': [p for n, p in model.named_parameters()
#                        if 'classifier' in n and not any(nd in n for nd in no_decay)],
#             'lr': head_lr,
#             'weight_decay': WEIGHT_DECAY,
#         },
#         {
#             'params': [p for n, p in model.named_parameters()
#                        if 'classifier' in n and any(nd in n for nd in no_decay)],
#             'lr': head_lr,
#             'weight_decay': 0.0,
#         },
#         # Backbone — lower learning rate
#         {
#             'params': [p for n, p in model.named_parameters()
#                        if 'classifier' not in n and not any(nd in n for nd in no_decay)],
#             'lr': base_lr,
#             'weight_decay': WEIGHT_DECAY,
#         },
#         {
#             'params': [p for n, p in model.named_parameters()
#                        if 'classifier' not in n and any(nd in n for nd in no_decay)],
#             'lr': base_lr,
#             'weight_decay': 0.0,
#         },
#     ]
#     return optimizer_params

# optimizer_params = get_optimizer_params(model, base_lr=LR, head_lr=LR * 20)
# optimizer = optim.AdamW(optimizer_params)


# total_steps = len(train_loader) * EPOCHS // GRAD_ACCUM_STEPS
# warmup_steps = len(train_loader) * WARMUP_EPOCHS // GRAD_ACCUM_STEPS

# def lr_lambda(current_step):
#     if current_step < warmup_steps:
#         return float(current_step) / float(max(1, warmup_steps))
#     progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
#     return max(0.0, 0.5 * (1.0 + np.cos(np.pi * progress)))

# scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# scaler = GradScaler()

In [15]:
# criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# best_f1 = 0.0
# best_model_path = os.path.join(OUTPUT_DIR, "model5_ast_best.pth")
# train_losses, val_losses = [], []
# train_f1s, val_f1s = [], []

# print("\n" + "="*60)
# print("TRAINING MODEL 5: Pretrained AST")
# print("="*60)
# print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}")
# print(f"Total steps: {total_steps}, Warmup steps: {warmup_steps}")

# for epoch in range(EPOCHS):
#     # --- TRAIN ---
#     model.train()
#     running_loss = 0.0
#     all_preds, all_labels = [], []
#     optimizer.zero_grad()

#     for batch_idx, (input_values, labels) in enumerate(train_loader):
#         input_values = input_values.to(DEVICE)
#         labels = labels.to(DEVICE)

#         with autocast():
#             outputs = model(input_values=input_values)
#             logits = outputs.logits
#             loss = criterion(logits, labels) / GRAD_ACCUM_STEPS

#         scaler.scale(loss).backward()

#         if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0:
#             scaler.unscale_(optimizer)
#             torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
#             scaler.step(optimizer)
#             scaler.update()
#             optimizer.zero_grad()
#             scheduler.step()

#         running_loss += loss.item() * GRAD_ACCUM_STEPS
#         preds = logits.argmax(dim=1).cpu().numpy()
#         all_preds.extend(preds)
#         all_labels.extend(labels.cpu().numpy())

#     train_loss = running_loss / len(train_loader)
#     train_f1 = f1_score(all_labels, all_preds, average='macro')
#     train_losses.append(train_loss)
#     train_f1s.append(train_f1)

#     # --- VALIDATE ---
#     model.eval()
#     val_loss = 0.0
#     all_val_preds, all_val_labels = [], []

#     with torch.no_grad():
#         for input_values, labels in val_loader:
#             input_values = input_values.to(DEVICE)
#             labels = labels.to(DEVICE)

#             with autocast():
#                 outputs = model(input_values=input_values)
#                 logits = outputs.logits
#                 loss = criterion(logits, labels)

#             val_loss += loss.item()
#             preds = logits.argmax(dim=1).cpu().numpy()
#             all_val_preds.extend(preds)
#             all_val_labels.extend(labels.cpu().numpy())

#     val_loss /= len(val_loader)
#     val_f1 = f1_score(all_val_labels, all_val_preds, average='macro')
#     val_losses.append(val_loss)
#     val_f1s.append(val_f1)

#     lr_now = optimizer.param_groups[0]['lr']
#     print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
#           f"Train Loss: {train_loss:.4f} F1: {train_f1:.4f} | "
#           f"Val Loss: {val_loss:.4f} F1: {val_f1:.4f} | "
#           f"LR: {lr_now:.7f}")

#     wandb.log({
#         "epoch": epoch + 1,
#         "train/loss": train_loss,
#         "train/macro_f1": train_f1,
#         "val/loss": val_loss,
#         "val/macro_f1": val_f1,
#         "lr": lr_now,
#         "best_val_f1": max(best_f1, val_f1),
#     })

#     if val_f1 > best_f1:
#         best_f1 = val_f1
#         torch.save({
#             'epoch': epoch,
#             'model_state_dict': model.state_dict(),
#             'val_f1': val_f1,
#         }, best_model_path)
#         print(f"  → Saved best model (F1: {val_f1:.4f})")

# print(f"\n{'='*60}")
# print(f"Best Validation Macro F1: {best_f1:.4f}")
# print(f"{'='*60}")

# wandb.finish()

In [16]:
# checkpoint = torch.load(best_model_path)
# model.load_state_dict(checkpoint['model_state_dict'])
# model.eval()

# all_val_preds, all_val_labels = [], []
# with torch.no_grad():
#     for input_values, labels in val_loader:
#         input_values = input_values.to(DEVICE)
#         with autocast():
#             outputs = model(input_values=input_values)
#         preds = outputs.logits.argmax(dim=1).cpu().numpy()
#         all_val_preds.extend(preds)
#         all_val_labels.extend(labels.numpy())

# report_str = classification_report(all_val_labels, all_val_preds, target_names=GENRES)
# print("\nClassification Report:")
# print(report_str)

In [17]:
# cm = confusion_matrix(all_val_labels, all_val_preds)
# plt.figure(figsize=(10, 8))
# sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
#             xticklabels=GENRES, yticklabels=GENRES)
# plt.title(f'Model 3 AST — Confusion Matrix (Macro F1: {best_f1:.4f})')
# plt.xlabel('Predicted')
# plt.ylabel('True')
# plt.tight_layout()
# plt.savefig(cm_path, dpi=150)
# plt.show()


# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
# ax1.plot(train_losses, label='Train Loss')
# ax1.plot(val_losses, label='Val Loss')
# ax1.set_title('Loss')
# ax1.set_xlabel('Epoch')
# ax1.legend()
# ax1.grid(True)

# ax2.plot(train_f1s, label='Train F1')
# ax2.plot(val_f1s, label='Val F1')
# ax2.axhline(y=0.8, color='r', linestyle='--', label='Target F1=0.8')
# ax2.set_title('Macro F1 Score')
# ax2.set_xlabel('Epoch')
# ax2.legend()
# ax2.grid(True)


# print("\nModel 5 training complete!")
# print(f"Best model saved to: {best_model_path}")


In [18]:
print("\n" + "="*60)
print("INFERENCE: Predicting test mashups")
print("="*60)

TEST_DIR = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups"
best_model_path = "/kaggle/input/models/sreekaranreddy2005/finetuned-ast/pytorch/default/2/model6_ast_best.pth"

checkpoint = torch.load(best_model_path, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Loaded best model (Val F1: {checkpoint['val_f1']:.4f})")


INFERENCE: Predicting test mashups
Loaded best model (Val F1: 0.8037)


In [19]:
def predict_audio_ast(filepath):
    """Load a test wav, chunk it, process through AST feature extractor, predict."""
    waveform, sr = torchaudio.load(filepath)

    
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    if sr != TARGET_SR:
        resampler = AT.Resample(sr, TARGET_SR)
        waveform = resampler(waveform)

    waveform_np = waveform.squeeze(0).numpy()

    chunk_samples = TARGET_SR * AUDIO_DURATION
    chunk_probs = []

    for start in range(0, len(waveform_np), chunk_samples):
        chunk = waveform_np[start:start + chunk_samples]


        if len(chunk) < chunk_samples // 2:
            continue

    
        if len(chunk) < chunk_samples:
            chunk = np.pad(chunk, (0, chunk_samples - len(chunk)))

        inputs = feature_extractor(
            chunk,
            sampling_rate=TARGET_SR,
            return_tensors="pt",
            padding="max_length",
            max_length=feature_extractor.max_length
        )
        input_values = inputs['input_values'].to(DEVICE)

        with torch.no_grad():
            with autocast():
                outputs = model(input_values=input_values)
            probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()
            chunk_probs.append(probs[0])

    if len(chunk_probs) == 0:
        return np.ones(NUM_CLASSES) / NUM_CLASSES

    return np.mean(chunk_probs, axis=0)

In [20]:
test_files = sorted([f for f in os.listdir(TEST_DIR) if f.endswith('.wav')])
print(f"Found {len(test_files)} test songs")

Found 3020 test songs


In [21]:
results = []
for i, filename in enumerate(test_files):
    filepath = os.path.join(TEST_DIR, filename)
    avg_probs = predict_audio_ast(filepath)

    predicted_idx = np.argmax(avg_probs)
    predicted_genre = IDX_TO_GENRE[predicted_idx]
    song_id = int(os.path.splitext(filename)[0].replace('song', ''))

    results.append({'id': song_id, 'genre': predicted_genre})

    if (i + 1) % 100 == 0 or (i + 1) == len(test_files):
        print(f"  Processed {i+1}/{len(test_files)} | {filename} -> {predicted_genre}")

results_df = pd.DataFrame(results)
results_df.to_csv("submission.csv", index=False)
print(f"\nSubmission saved to: submission.csv")

  Processed 100/3020 | song0100.wav -> country
  Processed 200/3020 | song0200.wav -> metal
  Processed 300/3020 | song0300.wav -> reggae
  Processed 400/3020 | song0400.wav -> metal
  Processed 500/3020 | song0500.wav -> pop
  Processed 600/3020 | song0600.wav -> rock
  Processed 700/3020 | song0700.wav -> hiphop
  Processed 800/3020 | song0800.wav -> pop
  Processed 900/3020 | song0900.wav -> blues
  Processed 1000/3020 | song1000.wav -> hiphop
  Processed 1100/3020 | song1100.wav -> pop
  Processed 1200/3020 | song1200.wav -> country
  Processed 1300/3020 | song1300.wav -> country
  Processed 1400/3020 | song1400.wav -> classical
  Processed 1500/3020 | song1500.wav -> blues
  Processed 1600/3020 | song1600.wav -> reggae
  Processed 1700/3020 | song1700.wav -> hiphop
  Processed 1800/3020 | song1800.wav -> hiphop
  Processed 1900/3020 | song1900.wav -> metal
  Processed 2000/3020 | song2000.wav -> rock
  Processed 2100/3020 | song2100.wav -> blues
  Processed 2200/3020 | song2200.wa